In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
portfolio = pd.read_csv("data/investment_portfolio_clean.csv")

portfolio.head()

In [ ]:
portfolio.shape

In [ ]:
portfolio.columns.tolist()

In [ ]:
portfolio[[
    "project_name",
    "capex_m",
    "criticality",
    "regulatory_driver",
    "capacity_benefit",
    "delivery_risk"
]].head(10)

In [ ]:
from src.validation import validate_portfolio

In [ ]:
validation = validate_portfolio(portfolio)

validation

In [ ]:
dirty_portfolio = pd.read_csv("data/investment_portfolio_dirty.csv")

dirty_validation = validate_portfolio(dirty_portfolio)

dirty_validation

In [ ]:
from src.scoring import *

In [ ]:
[name for name in dir() if "score" in name.lower() or "scenario" in name.lower()]

In [ ]:
SCENARIOS

In [ ]:
scored = score_portfolio(portfolio, scenario="Balanced")

scored[[
    "project_name",
    "priority_score",
    "rank"
]].head(10)

In [ ]:
top_project = scored.iloc[0]

top_project

In [ ]:
top_project[[
    "criticality",
    "regulatory_driver",
    "capacity_benefit",
    "customer_impact",
    "carbon_benefit",
    "project_maturity",
    "delivery_risk",
    "asset_condition_need",
    "priority_score"
]]

In [ ]:
manual_score = (
    7 * 0.20      # criticality
    + 8 * 0.20    # regulatory driver
    + 8 * 0.15    # capacity benefit
    + 5 * 0.10    # customer impact
    + 6 * 0.10    # carbon benefit
    + 10 * 0.10   # project maturity
    + 5.5 * 0.15  # asset condition need
    - 4 * 0.10    # delivery risk penalty
)

manual_score

In [ ]:
top_project[[
    "project_name",
    "capex_m",
    "priority_score",
    "rank"
]]

In [ ]:
reliability_scored = score_portfolio(
    portfolio,
    scenario="Reliability First"
)

reliability_scored[[
    "project_name",
    "priority_score",
    "rank"
]].head(10)

In [ ]:
project_name = top_project["project_name"]

balanced_result = scored[
    scored["project_name"] == project_name
][["project_name", "priority_score", "rank"]]

reliability_result = reliability_scored[
    reliability_scored["project_name"] == project_name
][["project_name", "priority_score", "rank"]]

balanced_result, reliability_result

In [ ]:
from src.optimiser import *

[name for name in dir() if "optim" in name.lower() or "portfolio" in name.lower()]

In [ ]:
from src.optimiser import optimise_portfolio

result = optimise_portfolio(scored, budget_m=250)

result.keys()

In [ ]:
{
    "budget_m": result["budget_m"],
    "funded_capex_m": result["funded_capex_m"],
    "remaining_budget_m": result["remaining_budget_m"],
    "selected_count": result["selected_count"],
    "deferred_count": result["deferred_count"],
    "portfolio_score": result["portfolio_score"],
}

In [ ]:
result["funded"][[
    "project_name",
    "capex_m",
    "priority_score",
    "rank",
    "mandatory"
]]

In [ ]:
result["deferred"][[
    "project_name",
    "capex_m",
    "priority_score",
    "rank",
    "mandatory"
]].head(10)

In [ ]:
result["funded"][["capex_m", "priority_score"]].sum()

In [ ]:
result["funded"]["mandatory"].value_counts()

In [ ]:
mandatory_capex = result["funded"].loc[
    result["funded"]["mandatory"] == True,
    "capex_m"
].sum()

mandatory_capex

In [ ]:
remaining_budget = 250
naive_selected = []

for _, project in scored.sort_values(
    ["rank", "priority_score"],
    ascending=[True, False]
).iterrows():

    if project["capex_m"] <= remaining_budget:
        naive_selected.append(project)
        remaining_budget -= project["capex_m"]

naive = pd.DataFrame(naive_selected)

naive[[
    "project_name",
    "capex_m",
    "priority_score",
    "rank"
]]

In [ ]:
{
    "projects_selected": len(naive),
    "capex": naive["capex_m"].sum(),
    "total_score": naive["priority_score"].sum(),
    "budget_remaining": remaining_budget
}

In [ ]:
{
    "projects_selected": result["selected_count"],
    "capex": result["funded_capex_m"],
    "total_score": result["portfolio_score"],
    "budget_remaining": result["remaining_budget_m"]
}

In [ ]:
naive["mandatory"].value_counts()

In [ ]:
mandatory_projects = set(
    scored.loc[scored["mandatory"] == True, "project_name"]
)

naive_projects = set(naive["project_name"])

mandatory_projects - naive_projects

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8001/assets",
    headers={"X-API-Key": "demo-secret-key"}
)

response.status_code

In [ ]:
payload = response.json()

payload.keys()

In [ ]:
payload["count"]

In [ ]:
payload["assets"][0]

In [ ]:
assets = pd.DataFrame(payload["assets"])

assets.head()

In [ ]:
assets.shape

In [ ]:
assets[[
    "asset_id",
    "asset_type",
    "condition_score",
    "inspection_date"
]].head(10)

In [ ]:
reconciled = portfolio.merge(
    assets,
    on="asset_id",
    how="left",
    suffixes=("_project", "_eam")
)

reconciled[[
    "project_name",
    "asset_id",
    "condition_score",
    "inspection_date"
]].head()

In [ ]:
{
    "portfolio_projects": len(portfolio),
    "eam_assets": len(assets),
    "matched": reconciled["condition_score"].notna().sum(),
    "unmatched": reconciled["condition_score"].isna().sum()
}

In [ ]:
reconciled.loc[
    reconciled["condition_score"].isna(),
    ["project_name", "asset_id"]
]

In [ ]:
reconciled = portfolio.merge(
    assets,
    on="asset_id",
    how="left",
    suffixes=("_project", "_eam"),
    indicator=True
)

reconciled["_merge"].value_counts()

In [ ]:
reconciled.loc[
    reconciled["_merge"] == "left_only",
    ["project_name", "asset_id"]
]

In [ ]:
reconciled.loc[
    (reconciled["_merge"] == "both")
    & (reconciled["condition_score"].isna()),
    ["project_name", "asset_id", "condition_score"]
]

In [ ]:
import numpy as np

reconciled["condition_data_status"] = np.select(
    [
        reconciled["_merge"] == "left_only",
        reconciled["condition_score"].isna(),
    ],
    [
        "missing_asset_record",
        "missing_condition_score",
    ],
    default="matched"
)

reconciled["condition_data_status"].value_counts()

In [ ]:
reconciled["asset_condition_need"] = np.where(
    reconciled["condition_score"].notna(),
    11 - reconciled["condition_score"],
    5.5
)

reconciled[[
    "project_name",
    "asset_id",
    "condition_score",
    "asset_condition_need",
    "condition_data_status"
]].tail(20)

In [ ]:
integrated_scored = score_portfolio(
    reconciled,
    scenario="Balanced"
)

integrated_scored[[
    "project_name",
    "condition_score",
    "asset_condition_need",
    "priority_score",
    "rank"
]].head(10)

In [ ]:
comparison = scored[[
    "project_name",
    "priority_score",
    "rank"
]].merge(
    integrated_scored[[
        "project_name",
        "condition_score",
        "asset_condition_need",
        "priority_score",
        "rank",
        "condition_data_status"
    ]],
    on="project_name",
    suffixes=("_before", "_after")
)

comparison["score_change"] = (
    comparison["priority_score_after"]
    - comparison["priority_score_before"]
)

comparison["rank_change"] = (
    comparison["rank_before"]
    - comparison["rank_after"]
)

comparison.sort_values(
    "rank_change",
    ascending=False
).head(10)

In [ ]:
comparison.sort_values(
    "rank_change",
    ascending=False
)[[
    "project_name",
    "condition_score",
    "asset_condition_need",
    "priority_score_before",
    "priority_score_after",
    "rank_before",
    "rank_after",
    "rank_change",
    "condition_data_status"
]].head(5)

In [ ]:
comparison.sort_values(
    "rank_change",
    ascending=True
)[[
    "project_name",
    "condition_score",
    "asset_condition_need",
    "priority_score_before",
    "priority_score_after",
    "rank_before",
    "rank_after",
    "rank_change",
    "condition_data_status"
]].head(5)

In [ ]:
openapi = requests.get(
    "http://127.0.0.1:8001/openapi.json"
).json()

list(openapi["paths"].keys())

In [ ]:
{
    path: list(methods.keys())
    for path, methods in openapi["paths"].items()
}

In [ ]:
openapi = requests.get(
    "http://127.0.0.1:8001/openapi.json"
).json()

{
    path: list(methods.keys())
    for path, methods in openapi["paths"].items()
}

In [ ]:
portfolio.loc[
    portfolio["project_name"] == "North Circuit Upgrade 15",
    ["project_name", "asset_id"]
]

In [ ]:
asset_id = portfolio.loc[
    portfolio["project_name"] == "North Circuit Upgrade 15",
    "asset_id"
].iloc[0]

asset_id

In [ ]:
current = requests.get(
    f"http://127.0.0.1:8001/assets/{asset_id}/condition",
    headers={"X-API-Key": "demo-secret-key"}
)

current.status_code, current.json()

In [ ]:
update = requests.patch(
    f"http://127.0.0.1:8001/assets/{asset_id}/condition",
    headers={"X-API-Key": "demo-secret-key"},
    json={
        "condition_score": 7,
        "inspection_date": "2026-10-28"
    }
)

update.status_code

In [ ]:
update.json()

In [ ]:
check = requests.get(
    f"http://127.0.0.1:8001/assets/{asset_id}/condition",
    headers={"X-API-Key": "demo-secret-key"}
)

check.json()

In [ ]:
response = requests.get(
    "http://127.0.0.1:8001/assets",
    headers={"X-API-Key": "demo-secret-key"}
)

assets_updated = pd.DataFrame(response.json()["assets"])

In [ ]:
assets_updated.loc[
    assets_updated["asset_id"] == "A115",
    ["asset_id", "condition_score", "inspection_date"]
]

In [ ]:
reconciled_updated = portfolio.merge(
    assets_updated,
    on="asset_id",
    how="left",
    suffixes=("_project", "_eam"),
    indicator=True
)

In [ ]:
reconciled_updated["condition_data_status"] = np.select(
    [
        reconciled_updated["_merge"] == "left_only",
        reconciled_updated["condition_score"].isna(),
    ],
    [
        "missing_asset_record",
        "missing_condition_score",
    ],
    default="matched"
)

In [ ]:
reconciled_updated["asset_condition_need"] = np.where(
    reconciled_updated["condition_score"].notna(),
    11 - reconciled_updated["condition_score"],
    5.5
)

In [ ]:
reconciled_updated.loc[
    reconciled_updated["project_name"] == "North Circuit Upgrade 15",
    [
        "project_name",
        "condition_score",
        "asset_condition_need",
        "condition_data_status"
    ]
]

In [ ]:
updated_scored = score_portfolio(
    reconciled_updated,
    scenario="Balanced"
)

In [ ]:
updated_scored.loc[
    updated_scored["project_name"] == "North Circuit Upgrade 15",
    [
        "project_name",
        "priority_score",
        "rank",
        "condition_score",
        "asset_condition_need"
    ]
]

In [ ]:
updated_result = optimise_portfolio(
    updated_scored,
    budget_m=250
)

project_name = "North Circuit Upgrade 15"

if project_name in set(updated_result["funded"]["project_name"]):
    print("FUNDED")
else:
    print("DEFERRED")


In [ ]:
updated_result["funded"].loc[
    updated_result["funded"]["project_name"] == project_name,
    ["project_name", "capex_m", "priority_score", "rank"]
]

In [ ]:
updated_result["deferred"].loc[
    updated_result["deferred"]["project_name"] == project_name,
    ["project_name", "capex_m", "priority_score", "rank"]
]

In [ ]:
pre_update_result = optimise_portfolio(
    integrated_scored,
    budget_m=250
)

pre_status = (
    "FUNDED"
    if "North Circuit Upgrade 15"
    in set(pre_update_result["funded"]["project_name"])
    else "DEFERRED"
)

pre_status

In [ ]:
post_update_result = optimise_portfolio(
    updated_scored,
    budget_m=250
)

post_status = (
    "FUNDED"
    if "North Circuit Upgrade 15"
    in set(post_update_result["funded"]["project_name"])
    else "DEFERRED"
)

pre_status, post_status

In [ ]:
pre_funded = set(
    pre_update_result["funded"]["project_name"]
)

post_funded = set(
    post_update_result["funded"]["project_name"]
)

dropped_from_portfolio = pre_funded - post_funded
added_to_portfolio = post_funded - pre_funded

dropped_from_portfolio, added_to_portfolio

In [ ]:
baseline_scored = score_portfolio(
    reconciled,
    scenario="Balanced"
)

baseline_result = optimise_portfolio(
    baseline_scored,
    budget_m=250
)

baseline_funded = set(
    baseline_result["funded"]["project_name"]
)

changes = []

for _, row in reconciled.iterrows():

    if pd.isna(row["condition_score"]):
        continue

    project_name = row["project_name"]
    original_condition = row["condition_score"]

    for test_condition in range(1, 11):

        if test_condition == original_condition:
            continue

        test_data = reconciled.copy()

        mask = test_data["project_name"] == project_name

        test_data.loc[
            mask,
            "condition_score"
        ] = test_condition

        test_data.loc[
            mask,
            "asset_condition_need"
        ] = 11 - test_condition

        test_scored = score_portfolio(
            test_data,
            scenario="Balanced"
        )

        test_result = optimise_portfolio(
            test_scored,
            budget_m=250
        )

        test_funded = set(
            test_result["funded"]["project_name"]
        )

        if test_funded != baseline_funded:

            changes.append({
                "project_name": project_name,
                "original_condition": original_condition,
                "test_condition": test_condition,
                "was_funded": project_name in baseline_funded,
                "now_funded": project_name in test_funded,
                "dropped_projects": baseline_funded - test_funded,
                "added_projects": test_funded - baseline_funded,
            })

In [ ]:
len(changes)

In [ ]:
pd.DataFrame(changes).head(20)

In [ ]:
response = requests.get(
    "http://127.0.0.1:8001/assets",
    headers={"X-API-Key": "demo-secret-key"}
)

response.raise_for_status()

assets_current = pd.DataFrame(
    response.json()["assets"]
)

assets_current.shape

In [ ]:
current_data = portfolio.merge(
    assets_current,
    on="asset_id",
    how="left",
    suffixes=("_project", "_eam"),
    indicator=True
)

current_data["condition_data_status"] = np.select(
    [
        current_data["_merge"] == "left_only",
        current_data["condition_score"].isna(),
    ],
    [
        "missing_asset_record",
        "missing_condition_score",
    ],
    default="matched"
)

current_data["asset_condition_need"] = np.where(
    current_data["condition_score"].notna(),
    11 - current_data["condition_score"],
    5.5
)

In [ ]:
current_data.loc[
    current_data["asset_id"] == "A115",
    [
        "project_name",
        "condition_score",
        "asset_condition_need"
    ]
]

In [ ]:
current_scored = score_portfolio(
    current_data,
    scenario="Balanced"
)

current_result = optimise_portfolio(
    current_scored,
    budget_m=250
)

current_funded = set(
    current_result["funded"]["project_name"]
)

In [ ]:
changes_current = []

for _, row in current_data.iterrows():

    if pd.isna(row["condition_score"]):
        continue

    project_name = row["project_name"]
    asset_id = row["asset_id"]
    original_condition = int(row["condition_score"])

    for test_condition in range(1, 11):

        if test_condition == original_condition:
            continue

        test_data = current_data.copy()

        mask = test_data["project_name"] == project_name

        test_data.loc[
            mask,
            "condition_score"
        ] = test_condition

        test_data.loc[
            mask,
            "asset_condition_need"
        ] = 11 - test_condition

        test_scored = score_portfolio(
            test_data,
            scenario="Balanced"
        )

        test_result = optimise_portfolio(
            test_scored,
            budget_m=250
        )

        test_funded = set(
            test_result["funded"]["project_name"]
        )

        if test_funded != current_funded:

            dropped = current_funded - test_funded
            added = test_funded - current_funded

            changes_current.append({
                "project_name": project_name,
                "asset_id": asset_id,
                "original_condition": original_condition,
                "test_condition": test_condition,
                "was_funded": project_name in current_funded,
                "now_funded": project_name in test_funded,
                "dropped_projects": ", ".join(sorted(dropped)),
                "added_projects": ", ".join(sorted(added)),
                "portfolio_changes": len(dropped) + len(added),
            })

In [ ]:
len(changes_current)

In [ ]:
changes_df = pd.DataFrame(changes_current)

changes_df.sort_values(
    ["portfolio_changes", "project_name", "test_condition"]
).head(20)

In [ ]:
project = "West SCADA / Control Upgrade 33"

current_scored.loc[
    current_scored["project_name"] == project,
    [
        "project_name",
        "asset_id",
        "condition_score",
        "asset_condition_need",
        "priority_score",
        "rank"
    ]
]

In [ ]:
"FUNDED" if project in current_funded else "DEFERRED"

In [ ]:
replacement = "East Protection System Modernisation 22"

"FUNDED" if replacement in current_funded else "DEFERRED"

In [ ]:
update = requests.patch(
    "http://127.0.0.1:8001/assets/A133/condition",
    headers={"X-API-Key": "demo-secret-key"},
    json={
        "condition_score": 5,
        "inspection_date": "2026-10-28"
    }
)

update.status_code, update.json()

In [ ]:
requests.get(
    "http://127.0.0.1:8001/assets/A133/condition",
    headers={"X-API-Key": "demo-secret-key"}
).json()

In [ ]:
response = requests.get(
    "http://127.0.0.1:8001/assets",
    headers={"X-API-Key": "demo-secret-key"}
)

response.raise_for_status()

assets_after = pd.DataFrame(
    response.json()["assets"]
)

In [ ]:
after_data = portfolio.merge(
    assets_after,
    on="asset_id",
    how="left",
    suffixes=("_project", "_eam"),
    indicator=True
)

after_data["condition_data_status"] = np.select(
    [
        after_data["_merge"] == "left_only",
        after_data["condition_score"].isna(),
    ],
    [
        "missing_asset_record",
        "missing_condition_score",
    ],
    default="matched"
)

after_data["asset_condition_need"] = np.where(
    after_data["condition_score"].notna(),
    11 - after_data["condition_score"],
    5.5
)

In [ ]:
after_scored = score_portfolio(
    after_data,
    scenario="Balanced"
)

after_result = optimise_portfolio(
    after_scored,
    budget_m=250
)

after_funded = set(
    after_result["funded"]["project_name"]
)

In [ ]:
after_scored.loc[
    after_scored["project_name"] == project,
    [
        "project_name",
        "condition_score",
        "asset_condition_need",
        "priority_score",
        "rank"
    ]
]

In [ ]:
{
    "West SCADA 33 before": "FUNDED" if project in current_funded else "DEFERRED",
    "West SCADA 33 after": "FUNDED" if project in after_funded else "DEFERRED",
    "East Protection 22 before": "FUNDED" if replacement in current_funded else "DEFERRED",
    "East Protection 22 after": "FUNDED" if replacement in after_funded else "DEFERRED",
}

In [ ]:
import pandas as pd

seed = pd.read_csv("data/assets.csv")

In [ ]:
seed.loc[
    seed["asset_id"] == "A115",
    ["condition_score", "inspection_date"]
] = [2, "2023-03-15"]

seed.loc[
    seed["asset_id"] == "A133",
    ["condition_score", "inspection_date"]
] = [8, "2026-09-15"]

In [ ]:
seed.loc[
    seed["asset_id"].isin(["A115", "A133"]),
    ["asset_id", "condition_score", "inspection_date"]
]

In [ ]:
seed.to_csv(
    "data/assets_seed.csv",
    index=False
)

In [ ]:
from pathlib import Path

Path("data/assets_seed.csv").exists()

In [ ]:
openapi = requests.get(
    "http://127.0.0.1:8001/openapi.json"
).json()

{
    path: list(methods.keys())
    for path, methods in openapi["paths"].items()
}

In [ ]:
reset = requests.post(
    "http://127.0.0.1:8001/admin/reset",
    headers={"X-API-Key": "demo-secret-key"}
)

reset.status_code, reset.json()

In [ ]:
(
    200,
    {
        "status": "reset",
        "records_restored": 39
    }
)

In [ ]:
check_a133 = requests.get(
    "http://127.0.0.1:8001/assets/A133/condition",
    headers={"X-API-Key": "demo-secret-key"}
)

check_a133.json()


In [ ]:
check_a115 = requests.get(
    "http://127.0.0.1:8001/assets/A115/condition",
    headers={"X-API-Key": "demo-secret-key"}
)

check_a115.json()

In [ ]:
from src.eam_client import reset_assets

reset_assets()

In [ ]:
from src.eam_client import update_asset_condition

update_asset_condition(
    asset_id="A133",
    condition_score=5,
    inspection_date="2026-10-28"
)

In [ ]:
reset_assets()

In [ ]:
from pathlib import Path

docs_dir = Path("data/docs")

for path in docs_dir.rglob("*"):
    if path.is_file():
        print(path)

In [ ]:
from pathlib import Path

print("Working directory:", Path.cwd())
print("data exists:", Path("data").exists())
print("data/docs exists:", Path("data/docs").exists())

if Path("data").exists():
    print("\nContents of data:")
    for path in Path("data").iterdir():
        print(" -", path)

In [ ]:
from pathlib import Path

documents_dir = Path("data/documents")

print("Documents folder exists:", documents_dir.exists())
print("\nDocuments:")

for path in documents_dir.rglob("*"):
    if path.is_file():
        print(" -", path)
    

In [ ]:
from pathlib import Path
import json

policy_path = Path("data/documents/customer_investment_policy.md")
manifest_path = Path("data/documents/source_manifest.json")

print("=== CUSTOMER INVESTMENT POLICY ===\n")
print(policy_path.read_text(encoding="utf-8"))

print("\n\n=== SOURCE MANIFEST ===\n")
print(
    json.dumps(
        json.loads(manifest_path.read_text(encoding="utf-8")),
        indent=2
    )
)

In [ ]:
from src.retrieval import build_retriever

retriever = build_retriever()

query = (
    "What should happen when asset condition data is missing "
    "or unavailable?"
)

results = retriever.search(query, top_k=4)

for i, hit in enumerate(results, start=1):
    print(f"\n--- RESULT {i} ---")
    print("Source:", hit["source"])
    print("Score:", hit["score"])
    print("Text:")
    print(hit["text"])

In [ ]:
from pathlib import Path

for path in Path("data/documents").iterdir():
    print(path.name)

In [ ]:
from collections import Counter
from src.retrieval import load_chunks, build_retriever

# Re-read the documents folder now that the Ofgem PDFs are present
chunks = load_chunks()

print("Total chunks:", len(chunks))
print("\nChunks by source:")
for source, count in Counter(chunk.source for chunk in chunks).items():
    print(f" - {source}: {count}")

# Build a fresh retriever over the expanded corpus
retriever = build_retriever()

query = (
    "What does Ofgem say about asset condition, network risk "
    "and the justification for investment?"
)

results = retriever.search(query, top_k=5)

print("\n=== RETRIEVAL RESULTS ===")

for i, hit in enumerate(results, start=1):
    print(f"\n--- RESULT {i} ---")
    print("Source:", hit["source"])
    print("Page:", hit["page"])
    print("Score:", hit["score"])
    print("Text:")
    print(hit["text"][:1000])

In [ ]:
query = (
    "What does Ofgem say about asset condition, network risk "
    "and the justification for investment?"
)

results = retriever.search(query, top_k=5)

for i, hit in enumerate(results, start=1):
    print(f"\n--- RESULT {i} ---")
    print("Source:", hit["source"])
    print("Page:", hit["page"])
    print("Score:", hit["score"])
    print("Text:")
    print(hit["text"][:1000])

In [ ]:
query = (
    "What evidence and assessment does Ofgem use to decide "
    "whether electricity transmission investment is justified?"
)

results = retriever.search(query, top_k=5)

for i, hit in enumerate(results, start=1):
    print(f"\n--- RESULT {i} ---")
    print("Source:", hit["source"])
    print("Page:", hit["page"])
    print("Score:", hit["score"])
    print("Text:")
    print(hit["text"][:1000])

In [ ]:
from src.tools import ToolRuntime, execute_tool

runtime = ToolRuntime(
    portfolio=portfolio_enriched,
    retriever=retriever,
)

result = execute_tool(
    runtime,
    "run_budget_scenario",
    {
        "budget_m": 250,
        "scenario": "Reliability First",
    },
)

result

In [ ]:
from src.data_loader import load_portfolio
from src.eam_client import fetch_assets, reset_assets
from src.scoring import enrich_with_assets

# Put the demo back into a known baseline state
reset_assets()

# Load the clean investment portfolio
portfolio = load_portfolio("clean")

# Fetch current asset data through the REST API
assets = fetch_assets()

# Reconcile the portfolio with the EAM asset data
portfolio_enriched = enrich_with_assets(
    portfolio,
    assets,
)

print("Portfolio projects:", len(portfolio))
print("EAM asset records:", len(assets))
print("Enriched projects:", len(portfolio_enriched))

In [ ]:
from src.tools import ToolRuntime, execute_tool

runtime = ToolRuntime(
    portfolio=portfolio_enriched,
    retriever=retriever,
)

result = execute_tool(
    runtime,
    "run_budget_scenario",
    {
        "budget_m": 250,
        "scenario": "Reliability First",
    },
)

result

In [ ]:
evidence = execute_tool(
    runtime,
    "search_documents",
    {
        "query": (
            "What evidence does Ofgem consider when deciding "
            "whether transmission investment is justified?"
        ),
        "top_k": 4,
    },
)

evidence